In [1]:
from dask.distributed import Client
client = Client(n_workers=4, memory_limit="auto")

In [2]:
import lsdb
import numpy as np

# Rubin ecdfs catalog with flux and mag
cat = lsdb.open_catalog('/Users/heather/data/ecdfs_flux_mag_cat')

In [3]:
# Note that mag and magErr columns already exist
# These were computed by a util in rubin_dash
cat.head(1)

coord_dec  coord_decErr  coord_ra  coord_raErr  \
_healpix_29                                                           
2528713666757650643 -28.196205           0.0  53.18721          0.0   

                     g_psfFlux  g_psfFluxErr   g_psfMag  g_psfMagErr  \
_healpix_29                                                            
2528713666757650643  1744349.5    228.920868  15.795917     0.000142   

                     i_psfFlux  i_psfFluxErr  ...   y_psfFlux  y_psfFluxErr  \
_healpix_29                                   ...                             
2528713666757650643  1685627.0   1071.237427  ...  2806465.75    591.650208   

                      y_psfMag  y_psfMagErr      yErr  z_psfFlux  \
_healpix_29                                                        
2528713666757650643  15.279601     0.000229  0.007965  2272363.5   

                     z_psfFluxErr   z_psfMag z_psfMagErr  \
_healpix_29                                                
2528713666757650643    496.028137  15.508805    0.000237   

                                                    objectForcedSource  
_healpix_29                                                             
2528713666757650643  [{coord_ra: 53.18721, coord_dec: -28.196205, v...  

[1 rows x 42 columns]

In [4]:
import numpy as np
import numpy.typing as npt

# AB definition is zp=8.9 for 1 Jy
MAG_AB_ZP_NJY = 8.9 + 2.5 * 9

def flux2mag(flux_njy: npt.ArrayLike, flux_err_njy: npt.ArrayLike | None) -> npt.ArrayLike | tuple[npt.ArrayLike, npt.ArrayLike]:
    """Converts flux (nJy) to magnitude, and flux error to magnitude error.

    Parameters
    ----------
    flux_njy: np.Array
        Flux values in nJy.
    flux_err_njy: np.Array, optional
        Flux error values in nJy.

    Returns
    -------
    mag: np.Array
        Magnitude values corresponding to flux inputs.
    mag_err: np.Array
        If flux_err_njy was passed, the corresponding magnitude errors.
        Otherwise, not returned.

    Raises
    ------
    ValueError, if flux_njy and flux_err_njy are not the same shape.
    ValueError, if a negative flux error is passed.
    """
    # Flux and flux_err must be same shape
    if (flux_err_njy is not None) and (np.shape(flux_njy) != np.shape(flux_err_njy)):
        raise ValueError(f"Inconsistent shapes. flux_njy: {np.shape(flux_njy)} flux_err_njy: {np.shape(flux_err_njy)}")

    # If flux is empty or all-NaN, return empty or all-NaN
    if (np.size(flux_njy) == 0) or (np.isnan(flux_njy).all()):
        mag = flux_njy
        if flux_err_njy is None:
            return mag
        return mag, mag

    # Convert zero and negative fluxes to NaN, will be NaN in output
    if any(flux_njy <= 0):
        flux_njy[flux_njy <= 0] = np.nan
    # Convert flux to magnitude, copied from lightcurvelynx
    mag = MAG_AB_ZP_NJY - 2.5 * np.log10(flux_njy)
    if (flux_err_njy is None):
        return mag
    
    # Compute mag error
    if (np.isnan(flux_err_njy).all()):
        return mag, flux_err_njy
    if np.min(np.nan_to_num(flux_err_njy)) < 0:
        raise ValueError("Negative flux error passed")
    # Skip mag error when flux is NaN
    if any(np.isnan(flux_njy)):
        flux_err_njy[np.isnan(flux_njy)] = np.nan
    # Magnitude error given by error propagation
    mag_err = 2.5 / np.log(10) * np.divide(flux_err_njy, flux_njy)
    return mag, mag_err


In [5]:
def flux_mapper(df):
    flux_njy = np.copy(df['g_psfFlux'].to_numpy())
    flux_err_njy = np.copy(df['g_psfFluxErr'].to_numpy())
    (mag, err) = flux2mag(flux_njy, flux_err_njy)
    df['g_psfMag_new'] = mag
    df['g_psfMagErr_new'] = err
    return df



In [8]:
from dask.dataframe.utils import make_meta
meta = make_meta(cat.head(1))
meta['g_psfMag_new'] = []
meta['g_psfMagErr_new'] = []
cat.map_partitions(flux_mapper, meta=meta).head(20)[['g_psfFlux', 'g_psfFluxErr', 'g_psfMag', 'g_psfMagErr', 'g_psfMag_new', 'g_psfMagErr_new']]

,g_psfFlux,g_psfFluxErr,g_psfMag,g_psfMagErr,g_psfMag_new,g_psfMagErr_new
_healpix_29,,,,,,
2528713666757650643,1744349.5,228.920868,15.795917,0.000142,15.795916,0.000142
2528713667097101852,1898.243652,8.923483,23.204121,0.005104,23.204121,0.005104
2528713668080668800,182.774277,8.64306,25.745213,0.051381,25.745213,0.051342
2528713668205595053,222.430527,8.622305,25.532015,0.042109,25.532013,0.042088
2528713668265055001,949.558655,8.800217,23.956196,0.010063,23.956196,0.010062
2528713668762683571,252.211792,8.696928,25.395586,0.037454,25.395586,0.037439
2528713668935615221,215.892044,8.618099,25.564409,0.043364,25.564407,0.043341
2528713669227541716,155.965134,8.602039,25.917431,0.059943,25.917431,0.059882
2528713669335851185,110.02491,8.570423,26.296272,0.084745,26.296272,0.084574
